# Modeling — **IndoBERT fine-tuning** (Colab/GPU) — *driver*

Driver tipis: clone repo → jalankan `src.modeling.train_indobert`. Pilih dataset di sel
**bagian 2** lewat `TAG`/`SUBSET` (default **balanced3k** = 1000/kelas). Artefak otomatis
ber-suffix tag (mis. `indobert_balanced3k_metrics.json`).

## 1. Clone repo (privat) + dependensi

Repo privat di GitHub → clone butuh **Personal Access Token** (scope `repo`).


In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (repo privat, scope repo): ')
![ -d jokowi_sentiment_project ] || git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git
%cd jokowi_sentiment_project
%pip install -q 'transformers>=4.40' torch 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv


## 2. Set MONGO_URI + jalankan training

Tempel `MONGO_URI` (butuh IP allowlist `0.0.0.0/0` agar Colab bisa konek). Skrip melakukan
split kanonik (urut `comment_id`, seed 42) **identik** SVM → test set sama → perbandingan
adil. Hyperparameter default = konfigurasi terbaik (4 epoch, LR 2e-5).


In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# === Pilih dataset ===
#   balanced 3k : TAG, SUBSET = 'balanced3k', 'outputs/labeling/balanced_3000.csv'
#   full 14k    : TAG, SUBSET = 'full14k', ''
TAG    = 'balanced3k'
SUBSET = 'outputs/labeling/balanced_3000.csv'

flags = f'--tag {TAG}' + (f' --subset {SUBSET}' if SUBSET else '')
# Tambah --model-out indobert_model bila ingin menyimpan bobot model.
!python -m src.modeling.train_indobert $flags

## 3. Lihat hasil + unduh artefak

Skrip menyimpan `indobert_metrics.json` + `indobert_test_confusion.png` di `outputs/reports/`.
**Penting:** unduh JSON-nya dan commit ke `outputs/reports/` repo lokal agar skrip SVM
(`train_svm_full14k`) bisa memuatnya untuk tabel perbandingan akhir.


In [ ]:
import json
from IPython.display import Image, display
suffix = '' if TAG == 'full14k' else f'_{TAG}'
m = json.load(open(f'outputs/reports/indobert{suffix}_metrics.json'))
print(json.dumps(m, ensure_ascii=False, indent=2))
display(Image(f'outputs/reports/indobert{suffix}_test_confusion.png'))

In [ ]:
# (Opsional) unduh artefak ke lokal untuk di-commit ke outputs/reports/
from google.colab import files
suffix = '' if TAG == 'full14k' else f'_{TAG}'
files.download(f'outputs/reports/indobert{suffix}_metrics.json')
files.download(f'outputs/reports/indobert{suffix}_test_confusion.png')

**Bandingkan 3 model.** Setelah `indobert_<TAG>_metrics.json` ada di `outputs/reports/`
(commit dari Colab atau taruh manual), jalankan di lokal:

```bash
python -m src.modeling.compare_models --tag balanced3k
```

→ `model_comparison_balanced3k.csv` + `model_comparison_balanced3k_accuracy.png`.